# Fast Iterative Method (FIM) su GPU

Implementazione del **Block FIM (Algoritmo 7)** della tesi su GPU, scritto in **Python con Numba** (`numba.cuda`: i kernel sono compilati per la GPU, come in CUDA C++).
È verificato e confrontato con il **Fast Marching (Algoritmo 3)** in **C++** sequenziale, come nella tesi (§4.1.2, speedup eq. 4.7).
Test: **sorgente puntiforme** in mezzo omogeneo (cap. 4.1), stencil a **4 vicini** (eq. 3.1) come nella tesi.
(La rotazione dello stencil, §2.6.2, è per ora disponibile solo nel Fast Marching.)

**Prima di tutto:** *Runtime → Cambia tipo di runtime → T4 GPU*, poi esegui le celle in ordine.

## Controllo GPU
Da eseguire all'inizio di ogni sessione Colab: deve comparire una **Tesla T4**. Se non c'è: *Runtime → Cambia tipo di runtime → T4 GPU*.

In [ ]:
!nvidia-smi

## Riferimento CPU – Fast Marching in C++

Il Fast Marching resta in **C++** (più veloce di una versione Python: lo speedup non viene falsato).
È compilato come **libreria condivisa** (`libfmm.so`) e chiamato da Python con `ctypes`:
- `solve_quadratic` → **Algoritmo 2** (eq. 3.1);
- `solve_node` → *g(x)*: 4 vicini, oppure assi + stencil ruotato;
- `fmm_cpu` → **Algoritmo 3** con min-heap. È `extern "C"` e lavora direttamente sull'array NumPy `u`.

In [ ]:
%%writefile fmm.cpp
// fmm.cpp: Fast Marching (Algoritmo 3 della tesi) in C++, compilato come libreria
// condivisa (libfmm.so) e chiamato da Python con ctypes.
// E' il riferimento della tesi (§4.1.2): verifica del FIM su GPU e speedup (eq. 4.7).
#include <cmath>
#include <vector>
#include <queue>
#include <functional>

// Valore "infinito": nodo non ancora raggiunto dal fronte d'onda.
// Usiamo un numero finito molto grande (non std::numeric_limits<float>::infinity())
// cosi' che a - b tra due INF faccia 0 e non NaN.
const float INF = 1e30f;

const float SQRT2 = 1.41421356f;

// Spostamenti (di, dj) verso i vicini.
// I primi 4 sono quelli degli assi (sinistra, destra, sotto, sopra);
// gli ultimi 4 sono i diagonali, usati solo con la rotazione dello stencil.
const int DI[8] = {  0, 0, -1, 1,   -1, -1,  1, 1 };
const int DJ[8] = { -1, 1,  0, 0,   -1,  1, -1, 1 };
//   (u - a)^2 + (u - b)^2 = (F h)^2
// a = minimo dei vicini lungo x, b = minimo dei vicini lungo y.
float solve_quadratic(float a, float b, float F, float h)
{
    float Fh = F * h;
    if (fabsf(a - b) <= Fh)      // il fronte arriva da entrambe le direzioni
        return 0.5f * (a + b + sqrtf(2.0f * Fh * Fh - (a - b) * (a - b)));
    return fminf(a, b) + Fh;     // il fronte arriva da una sola direzione
}

// g(x) della tesi: calcola il nuovo valore nel nodo (i, j).
// Legge i 4 vicini (INF se fuori dal dominio) e chiama l'Algoritmo 2.
// Con rotated = true applica anche la rotazione dello stencil (tesi, §2.6.2):
// lo stesso Algoritmo 2 sui vicini diagonali con passo sqrt(2) h, e si tiene il minimo.
// Restituisce min(u_vecchio, u_nuovo): u puo' solo diminuire (eq. 3.2).
// ---------------------------------------------------------------------------
float solve_node(const float* u, const float* F,
                                     int i, int j, int N, float h, bool rotated)
{
    float left  = (j > 0)     ? u[i * N + (j - 1)] : INF;
    float right = (j < N - 1) ? u[i * N + (j + 1)] : INF;
    float down  = (i > 0)     ? u[(i - 1) * N + j] : INF;
    float up    = (i < N - 1) ? u[(i + 1) * N + j] : INF;

    float a = fminf(left, right);
    float b = fminf(down, up);

    int idx = i * N + j;
    float q = solve_quadratic(a, b, F[idx], h);           // stencil sugli assi

    if (rotated) {
        // Stencil ruotato di pi/4: i "nuovi assi" sono le due diagonali.
        float down_left  = (i > 0 && j > 0)         ? u[(i - 1) * N + (j - 1)] : INF;
        float up_right   = (i < N - 1 && j < N - 1) ? u[(i + 1) * N + (j + 1)] : INF;
        float up_left    = (i < N - 1 && j > 0)     ? u[(i + 1) * N + (j - 1)] : INF;
        float down_right = (i > 0 && j < N - 1)     ? u[(i - 1) * N + (j + 1)] : INF;

        float a_rot = fminf(down_left, up_right);         // minimo lungo la diagonale "in salita a destra"
        float b_rot = fminf(up_left, down_right);         // minimo lungo la diagonale "in salita a sinistra"
        float q_rot = solve_quadratic(a_rot, b_rot, F[idx], SQRT2 * h);

        q = fminf(q, q_rot);                              // causalita': vince il minore
    }
    return fminf(u[idx], q);
}

// Algoritmo 3 (Fast Marching), versione sequenziale.

// Stato dei nodi:  LONTANO (non raggiunto), ATTIVO (nella Narrow Band), NOTO (congelato).

// extern "C": il nome resta "fmm_cpu" anche dopo la compilazione, cosi' Python
// (ctypes) lo trova. u e F sono gli array NumPy float32 N x N passati come puntatori:
// il FMM lavora direttamente sull'array u di Python, senza copie.
// ---------------------------------------------------------------------------
extern "C" void fmm_cpu(float* u, const float* F, int N, float h, int src, bool rotated)
{
    enum { LONTANO = 0, ATTIVO = 1, NOTO = 2 };
    std::vector<char> state((size_t)N * N, LONTANO);

    typedef std::pair<float, int> Entry;                       // (u, indice del nodo)
    std::priority_queue<Entry, std::vector<Entry>, std::greater<Entry>> heap;

    heap.push(Entry(u[src], src));                             // la sorgente e' il primo Trial
    state[src] = ATTIVO;
    int n_nb = rotated ? 8 : 4;

    while (!heap.empty()) {
        Entry e = heap.top();                                  // Trial <- radice del min-heap
        heap.pop();                                            // rimuovi Trial da A
        int x = e.second;
        if (state[x] == NOTO || e.first > u[x]) continue;      // coppia scaduta: ignorala
        state[x] = NOTO;                                       // aggiungi Trial ai Noti

        int i = x / N, j = x % N;
        for (int k = 0; k < n_nb; ++k) {                       // foreach x_nb adiacente a Trial
            int ni = i + DI[k], nj = j + DJ[k];
            if (ni < 0 || ni >= N || nj < 0 || nj >= N) continue;
            int nb = ni * N + nj;
            if (state[nb] == NOTO) continue;                   // if x_nb non e' Noto

            float q = solve_node(u, F, ni, nj, N, h, rotated);
            if (q < u[nb]) {                                   // u(x_nb) <- g(x_nb)
                u[nb] = q;
                state[nb] = ATTIVO;                            // aggiungi x_nb ad A
                heap.push(Entry(q, nb));
            }
        }
    }
}

In [ ]:
!g++ -O3 -shared -fPIC fmm.cpp -o libfmm.so

In [ ]:
import ctypes
import numpy as np

# Libreria C++ compilata nella cella precedente
_lib = ctypes.CDLL("./libfmm.so")

# Tipi degli argomenti di fmm_cpu(float* u, const float* F, int N, float h, int src, bool rotated).
# ndpointer: ctypes passa al C++ il puntatore ai dati dell'array NumPy
# (deve essere float32 e "C_CONTIGUOUS", cioe' salvato riga per riga come in C).
_array_f32 = np.ctypeslib.ndpointer(dtype=np.float32, flags="C_CONTIGUOUS")
_lib.fmm_cpu.argtypes = [_array_f32, _array_f32, ctypes.c_int, ctypes.c_float,
                         ctypes.c_int, ctypes.c_bool]
_lib.fmm_cpu.restype = None


def fmm(u, F, h, src, rotated):
    """Fast Marching in C++ (Algoritmo 3). Modifica u sul posto.
    u, F: array N x N float32; src = (riga, colonna) della sorgente."""
    N = u.shape[0]
    _lib.fmm_cpu(u, F, N, h, src[0] * N + src[1], rotated)

## FIM – Block Fast Iterative Method (Algoritmo 7) su GPU con Numba

La griglia N×N è divisa in **blocchi di 8×8 nodi**; la lista A contiene **blocchi**, e ogni blocco attivo è elaborato da un **blocco CUDA di 8×8 thread** (un thread per nodo).

| Algoritmo 7 | Funzione |
|---|---|
| Step 1: foreach b ∈ A: `niter` iterazioni locali, Cb(b) = reduction(Cn(b)) | `update_active_tiles` → `process_tile(niter)` |
| Step 2: vicini dei blocchi convergenti: 1 iterazione + reduction | `update_neighbor_tiles` (*gather*) → `process_tile(1)` |
| Step 3: A = { b : Cb(b) = falso } | `build_active_list` (`cuda.atomic.add`) |

- `@cuda.jit` → **kernel** (lanciato dalla CPU con `kernel[blocchi, thread](...)`); `@cuda.jit(device=True)` → funzione chiamata dai kernel.
- `process_tile`: tile + **halo** in **shared memory** (`cuda.shared.array`) → `niter` iterazioni con `cuda.syncthreads()` → scrittura in memoria globale → **reduction AND** (fig. 3.7).
- Tutti i valori sono `float32`, come in C++: le costanti sono scritte `float32(...)`, altrimenti Python userebbe `float64`.

In [ ]:
import math
import warnings
import numpy as np
from numba import cuda, float32, int32
from numba.core.errors import NumbaPerformanceWarning

# Numba avvisa quando un kernel e' lanciato con pochi blocchi (succede nei primi giri,
# quando A contiene 1-2 blocchi): non e' un errore, quindi nascondiamo l'avviso.
warnings.simplefilter("ignore", NumbaPerformanceWarning)

# Un blocco b della tesi = TILE x TILE nodi = un blocco CUDA di TILE x TILE thread.
# TILE * TILE deve essere una potenza di 2 per la reduction (fig. 3.7).
TILE = 8
# Dimensioni degli array in shared memory: devono essere COSTANTI note alla compilazione.
# Vanno calcolate qui: dentro un kernel, "TILE + 2" per Numba sarebbe un intero qualsiasi.
TILE_HALO = TILE + 2         # lato della tile con il bordo (halo): 1 + 8 + 1 = 10
TILE2 = TILE * TILE          # nodi (e thread) per blocco: 64
INF = float32(1e30)          # nodo non ancora raggiunto
HALF = float32(0.5)          # costanti float32: un 0.5 scritto cosi' sarebbe float64
TWO = float32(2.0)


# ===========================================================================
# Algoritmo 2 della tesi: soluzione dell'equazione quadratica (3.1).
# a = minimo dei vicini lungo x, b = minimo dei vicini lungo y.
# ===========================================================================
@cuda.jit(device=True)
def solve_quadratic(a, b, F, h):
    Fh = F * h
    if abs(a - b) <= Fh:                 # il fronte arriva da entrambe le direzioni
        return HALF * (a + b + math.sqrt(TWO * Fh * Fh - (a - b) * (a - b)))
    return min(a, b) + Fh                # il fronte arriva da una sola direzione


# ===========================================================================
# Algoritmo 2 applicato al nodo (y, x) della tile in shared memory.
# La tile ha un bordo (halo) di 1 nodo su ogni lato: il nodo (ty, tx) del
# blocco sta in s[ty + 1, tx + 1]. Fuori dal dominio l'halo vale INF.
# ===========================================================================
@cuda.jit(device=True)
def solve_in_tile(s, y, x, F, h):
    a = min(s[y, x - 1], s[y, x + 1])            # sinistra, destra  (asse x)
    b = min(s[y - 1, x], s[y + 1, x])            # sotto, sopra      (asse y)
    q = solve_quadratic(a, b, F, h)
    return min(s[y, x], q)                       # u puo' solo diminuire


# ===========================================================================
# Aggiornamento di UN blocco (tile) da parte di un blocco CUDA:
#   for i = 0 to niter:  (U(b), Cn(b)) = soluzione di g(b)
#   Cb(b) = reduction(Cn(b))
# s_u e s_c sono gli array in shared memory, creati dal kernel che la chiama.
# Deve essere chiamata da TUTTI i thread del blocco (contiene syncthreads).
# ===========================================================================
@cuda.jit(device=True)
def process_tile(tile, niter, u, F, h, eps, src_i, src_j, Cb_out, s_u, s_c):
    N = u.shape[0]
    nbx = (N + TILE - 1) // TILE
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    tid = ty * TILE + tx                         # indice del thread nel blocco: 0 .. TILE2-1

    bi = tile // nbx                             # riga e colonna del blocco
    bj = tile % nbx
    i = bi * TILE + ty                           # nodo globale di questo thread
    j = bj * TILE + tx
    inside = i < N and j < N                     # l'ultimo blocco puo' sporgere dal dominio
    fixed = (not inside) or (i == src_i and j == src_j)   # fuori dominio o sorgente

    # ---- 1. memoria globale -> shared (tile + halo) ----
    # 100 celle e 64 thread: ogni thread copia la cella k, poi k + 64.
    for k in range(tid, TILE_HALO * TILE_HALO, TILE2):
        sy = k // TILE_HALO
        sx = k % TILE_HALO
        gi = bi * TILE + sy - 1                  # -1: la riga 0 di s_u e' quella sopra il blocco
        gj = bj * TILE + sx - 1
        if 0 <= gi < N and 0 <= gj < N:
            s_u[sy, sx] = u[gi, gj]
        else:
            s_u[sy, sx] = INF
    f = F[i, j] if inside else float32(1.0)
    cuda.syncthreads()                           # la tile e' completa prima di usarla

    # ---- 2. niter iterazioni locali in shared memory ----
    conv = True
    for it in range(niter):
        p = s_u[ty + 1, tx + 1]
        q = p if fixed else solve_in_tile(s_u, ty + 1, tx + 1, f, h)
        cuda.syncthreads()                       # tutti hanno letto i vicini...
        s_u[ty + 1, tx + 1] = q
        cuda.syncthreads()                       # ...prima che qualcuno li sovrascriva
        conv = abs(p - q) < eps                  # Cn del nodo all'ultima iterazione

    # ---- 3. shared -> memoria globale (solo i nodi del blocco, non l'halo) ----
    if inside:
        u[i, j] = s_u[ty + 1, tx + 1]

    # ---- 4. reduction AND (fig. 3.7): 64 -> 32 -> ... -> 1 ----
    s_c[tid] = 1 if conv else 0
    cuda.syncthreads()
    stride = TILE2 // 2
    while stride > 0:
        if tid < stride:
            s_c[tid] = s_c[tid] & s_c[tid + stride]
        cuda.syncthreads()
        stride //= 2
    if tid == 0:
        Cb_out[tile] = s_c[0]


# ===========================================================================
# KERNEL Step 1 dell'Algoritmo 7: aggiornamento dei blocchi attivi.
# Si lanciano SOLO i blocchi attivi: il blocco CUDA k elabora il blocco lst[k].
# ===========================================================================
@cuda.jit
def update_active_tiles(lst, u, F, h, eps, src_i, src_j, niter, Cb1):
    s_u = cuda.shared.array((TILE_HALO, TILE_HALO), float32)   # tile + halo
    s_c = cuda.shared.array(TILE2, int32)                       # flag Cn per la reduction
    process_tile(lst[cuda.blockIdx.x], niter, u, F, h, eps, src_i, src_j, Cb1, s_u, s_c)


# ===========================================================================
# KERNEL Step 2 dell'Algoritmo 7: aggiornamento dei blocchi vicini ("gather").
# Ogni blocco guarda il PROPRIO stato: "un mio blocco vicino e' appena converso?".
# Un blocco ancora attivo e non convergente NON viene testato (resta in A).
# ===========================================================================
@cuda.jit
def update_neighbor_tiles(u, F, h, eps, src_i, src_j, active, Cb1, Cb):
    s_u = cuda.shared.array((TILE_HALO, TILE_HALO), float32)
    s_c = cuda.shared.array(TILE2, int32)
    nbx = cuda.gridDim.x                         # un blocco CUDA per ogni blocco della griglia
    bi = cuda.blockIdx.y
    bj = cuda.blockIdx.x
    tile = bi * nbx + bj
    first = cuda.threadIdx.x == 0 and cuda.threadIdx.y == 0   # un solo thread scrive Cb

    # Caso 1: blocco in A e non convergente -> resta in A, nessun test.
    if active[tile] and not Cb1[tile]:
        if first:
            Cb[tile] = 0
        return

    # Caso 2: c'e' un blocco vicino appena converso (in A e Cb1 = vero)?
    # I 4 vicini sono a distanza 1 (sinistra/destra) e nbx (sotto/sopra) nella numerazione.
    left  = bj > 0       and active[tile - 1] != 0   and Cb1[tile - 1] != 0
    right = bj < nbx - 1 and active[tile + 1] != 0   and Cb1[tile + 1] != 0
    down  = bi > 0       and active[tile - nbx] != 0 and Cb1[tile - nbx] != 0
    up    = bi < nbx - 1 and active[tile + nbx] != 0 and Cb1[tile + nbx] != 0
    near_converged = left or right or down or up

    # Tutti i thread hanno calcolato le stesse condizioni: escono tutti o restano tutti.
    if not near_converged:
        if first:
            Cb[tile] = 1                         # nessuna novita' dai vicini: convergente
        return
    process_tile(tile, 1, u, F, h, eps, src_i, src_j, Cb, s_u, s_c)   # test: 1 iterazione


# ===========================================================================
# KERNEL Step 3 dell'Algoritmo 7: nuova lista dei blocchi attivi.
# atomic.add incrementa count[0] e restituisce il valore PRIMA dell'incremento,
# in modo indivisibile: ogni blocco attivo ottiene una posizione diversa.
# ===========================================================================
@cuda.jit
def build_active_list(Cb, active, lst, count):
    t = cuda.grid(1)                             # = blockIdx.x * blockDim.x + threadIdx.x
    if t >= Cb.shape[0]:
        return
    act = Cb[t] == 0
    active[t] = 1 if act else 0
    if act:
        lst[cuda.atomic.add(count, 0, 1)] = t


# ===========================================================================
# Ciclo dell'Algoritmo 7 sull'host.
# Restituisce (soluzione u, iterazioni globali, tempo GPU in ms).
# ===========================================================================
def block_fim(u0, F, h, src, eps, niter):
    N = u0.shape[0]
    nbx = (N + TILE - 1) // TILE                 # blocchi per lato
    n_tiles = nbx * nbx

    # Stato iniziale: tutti i blocchi "convergenti" tranne quello della sorgente
    Cb0 = np.ones(n_tiles, dtype=np.int32)
    Cb0[(src[0] // TILE) * nbx + src[1] // TILE] = 0

    # Copie sulla GPU
    d_u = cuda.to_device(u0)
    d_F = cuda.to_device(F)
    d_Cb = cuda.to_device(Cb0)
    d_Cb1 = cuda.to_device(np.zeros(n_tiles, dtype=np.int32))
    d_active = cuda.device_array(n_tiles, dtype=np.int32)
    d_list = cuda.device_array(n_tiles, dtype=np.int32)
    d_count = cuda.device_array(1, dtype=np.int32)
    zero = np.zeros(1, dtype=np.int32)
    n_active = np.zeros(1, dtype=np.int32)

    h = float32(h)
    eps = float32(eps)
    list_threads = 256
    list_blocks = (n_tiles + list_threads - 1) // list_threads

    start, stop = cuda.event(), cuda.event()
    start.record()

    it = 0
    while True:
        # Step 3 (fatto per primo): costruisce A dai Cb correnti
        d_count.copy_to_device(zero)
        build_active_list[list_blocks, list_threads](d_Cb, d_active, d_list, d_count)
        d_count.copy_to_host(n_active)           # aspetta il kernel
        if n_active[0] == 0:                     # A vuota: fine
            break
        it += 1

        # Step 1: solo i blocchi attivi -> n_active blocchi CUDA
        update_active_tiles[int(n_active[0]), (TILE, TILE)](
            d_list, d_u, d_F, h, eps, src[0], src[1], niter, d_Cb1)

        # Step 2: blocchi vicini a quelli appena convergenti
        update_neighbor_tiles[(nbx, nbx), (TILE, TILE)](
            d_u, d_F, h, eps, src[0], src[1], d_active, d_Cb1, d_Cb)

    stop.record()
    stop.synchronize()
    return d_u.copy_to_host(), it, start.elapsed_time(stop)

### Test: Block FIM (GPU) vs Fast Marching (CPU)
Per ogni N: iterazioni globali, max|GPU−FMM| (correttezza), errore relativo (eq. 4.2), tempi e **speedup** t_FMM / t_FIM (eq. 4.7).
Ogni tempo è il **minimo di 3 misure**. La prima chiamata dei kernel li compila (qualche secondo): viene fatta prima, fuori dalle misure.

In [ ]:
import time


def relative_error(u, h):
    """Errore relativo, eq. (4.2): ||u - U|| / ||U||, con U = sqrt(x^2 + y^2)."""
    x = np.linspace(-1.0, 1.0, u.shape[0])
    X, Y = np.meshgrid(x, x)
    U = np.sqrt(X**2 + Y**2)
    return np.linalg.norm(u - U) / np.linalg.norm(U)


def run_test(niter, filename, sizes=(101, 401, 1001, 2001, 4001), eps=1e-6, repeat=3):
    # Ogni misura di tempo e' ripetuta 'repeat' volte e si tiene la minima:
    # una singola misura puo' essere disturbata da altri processi sulla macchina.
    print(f"===== Stencil assi (4 vicini), TILE = {TILE}, niter = {niter} =====")
    print("   N  iter glob.  max|GPU-FMM|  errore rel.  t FMM CPU [ms]  t FIM GPU [ms]  speedup")
    for N in sizes:
        h = 2.0 / (N - 1)                                  # dominio [-1, 1]^2
        src = (N // 2, N // 2)                             # nodo centrale -> (0, 0)
        F = np.ones((N, N), dtype=np.float32)              # mezzo omogeneo
        u0 = np.full((N, N), INF, dtype=np.float32)
        u0[src] = 0.0

        # Fast Marching su CPU (C++): riferimento della tesi, stesso stencil a 4 vicini
        ms_fmm = float("inf")
        for _ in range(repeat):
            u_fmm = u0.copy()                              # ogni volta si riparte da zero
            t0 = time.perf_counter()
            fmm(u_fmm, F, h, src, rotated=False)
            ms_fmm = min(ms_fmm, (time.perf_counter() - t0) * 1e3)

        # Block FIM su GPU
        ms_gpu = float("inf")
        for _ in range(repeat):
            u_gpu, it, ms = block_fim(u0, F, h, src, eps, niter)
            ms_gpu = min(ms_gpu, ms)

        max_diff = np.abs(u_gpu - u_fmm).max()
        print(f"{N:5d}  {it:9d}     {max_diff:.2e}    {relative_error(u_gpu, h):.3e}"
              f"  {ms_fmm:14.2f}  {ms_gpu:14.2f}  {ms_fmm / ms_gpu:7.2f}")

        if N == 101:
            u_gpu.tofile(filename)


niter = 2 * TILE                           # iterazioni locali per blocco

# Riscaldamento: la prima chiamata compila i kernel (qualche secondo), fuori dalle misure
block_fim(np.full((51, 51), INF, dtype=np.float32), np.ones((51, 51), dtype=np.float32),
          0.04, (25, 25), 1e-6, niter)

run_test(niter, "block_fim.bin")
print("\nSoluzione GPU N = 101 salvata in block_fim.bin")

### Visualizzazione (N = 101)
A sinistra: curve di livello del Block FIM (continue) sovrapposte a quelle analitiche (tratteggiate), come in fig. 4.1.
A destra: errore |u − U| in ogni nodo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

N = 101
u = np.fromfile('block_fim.bin', dtype=np.float32).reshape((N, N))   # riga i = asse y, colonna j = asse x
x = np.linspace(-1, 1, N)
X, Y = np.meshgrid(x, x)
U = np.sqrt(X**2 + Y**2)                                             # soluzione analitica (eq. 4.3)

fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))

levels = np.arange(0.2, 1.5, 0.2)
c1 = ax[0].contour(X, Y, u, levels=levels, colors='tab:blue')
ax[0].contour(X, Y, U, levels=levels, colors='k', linestyles='--')
ax[0].clabel(c1, fmt='%.1f', fontsize=8)
ax[0].plot(0, 0, 'ro')
ax[0].set_aspect('equal')
ax[0].set_title('Curve di livello: Block FIM (blu) vs analitica (tratteggio)')

im = ax[1].imshow(np.abs(u - U), origin='lower', extent=[-1, 1, -1, 1], cmap='viridis')
fig.colorbar(im, ax=ax[1], label='|u − U|')
ax[1].set_title('Errore assoluto')

for a in ax:
    a.set_xlabel('x [u.a.]'); a.set_ylabel('y [u.a.]')

plt.tight_layout()
plt.show()